In [1]:
import skimage as sk
from skimage.filters import gaussian
from io import BytesIO
from wand.image import Image as WandImage
from wand.api import library as wandlibrary
import wand.color as WandColor
import ctypes
from PIL import Image as PILImage
import cv2
import numpy as np
import os
from tqdm import tqdm

In [2]:
SRC_PATH = "../../data/images/test/normal"
DES_PATH = "../../data/images/transform_test"

In [3]:
def shot_noise(x, severity=1):
    c = [60, 25, 12, 5, 3][severity - 1]

    x = np.array(x, dtype=np.float32) / 255.0 
    x_noisy = np.random.poisson(x * c) / c
    x_noisy = np.clip(x_noisy, 0, 1) * 255.0

    return x_noisy.astype(np.uint8)

def brightness(x, severity=1):
    c = [.1, .2, .3, .4, .5][severity - 1]

    x = np.array(x) / 255.
    x = sk.color.rgb2hsv(x)
    x[:, :, 2] = np.clip(x[:, :, 2] + c, 0, 1)
    x = sk.color.hsv2rgb(x)
    x = np.clip(x * 255.0, 0, 255).astype(np.uint8)

    return x

def jpeg_compression(x, severity=1):
    c = [25, 18, 15, 10, 7][severity - 1]

    output = BytesIO()
    x.save(output, 'JPEG', quality=c)
    output.seek(0)
    img = PILImage.open(output).convert('RGB')

    return np.array(img, dtype=np.uint8)

In [4]:
def motion_blur(img, severity=1):
    c = [(10, 3), (15, 5), (15, 8), (15, 12), (20, 15)][severity - 1]
    
    radius, sigma = c
    k = radius
    kernel = np.zeros((k, k), dtype=np.float32)
    kernel[k//2, :] = 1.0
    angle = np.random.uniform(-45, 45)
    rot_mat = cv2.getRotationMatrix2D((k/2 - 0.5, k/2 - 0.5), angle, 1.0)
    kernel = cv2.warpAffine(kernel, rot_mat, (k, k))
    kernel = cv2.GaussianBlur(kernel, (0, 0), sigma)
    kernel /= kernel.sum()
    img_np = np.array(img, dtype=np.float32)
    blurred = cv2.filter2D(img_np, -1, kernel)

    return np.clip(blurred, 0, 255).astype(np.uint8)

In [5]:
DISTORTIONS = {
    # "motion_blur": motion_blur,
    # "shot_noise": shot_noise,
    "jpeg": jpeg_compression,
    "brightness": brightness,
}

def distort_image(src_path, des_path, distortion, severity=1):
    if distortion not in DISTORTIONS:
        raise ValueError(f"Unknown distortion: {distortion}\n")
    img = PILImage.open(src_path).convert('RGB')
    distorted = DISTORTIONS[distortion](img, severity)
    PILImage.fromarray(distorted).save(des_path)

In [6]:
def transform():
    files = [f for f in os.listdir(SRC_PATH) if os.path.isfile(os.path.join(SRC_PATH, f))]
    os.makedirs(DES_PATH, exist_ok=True)
    for distortion in DISTORTIONS:
        for severity in range(1,6):
            output_folder = os.path.join(DES_PATH, distortion, severity.__str__())
            os.makedirs(output_folder, exist_ok=True)
            for file in tqdm(files, desc = f"Transform {distortion} in severity {severity}"):
                output_path = os.path.join(output_folder, file)
                input_path = os.path.join(SRC_PATH, file)
                distort_image(input_path, output_path, distortion, severity)

In [7]:
transform()

Transform brightness in severity 5: 100%|██████████| 20000/20000 [5:32:32<00:00,  1.00it/s]     
